# Analise Exploratória de Dados

## Objetivo desta etapa

O objetivo desta etapa não é gerar conclusões finais de negócio, mas entender a estrutura dos dados e como se comportam no negócio para evitar análises incorretas.

Esperamos definir:

- a granularidade das tabelas;
- as chaves de relacionamento entre elas;
- a unidade de análise principal do projeto;
- os principais cuidados antes da modelagem dos dados.

### Imports

In [ ]:
import pandas as pd
PATH_DATA = "../data/raw/"

### Importação dos dados

In [ ]:
customers = pd.read_csv(
    PATH_DATA + "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    PATH_DATA + "olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    PATH_DATA + "olist_order_items_dataset.csv"
)

products = pd.read_csv(
    PATH_DATA + "olist_products_dataset.csv"
)

payments = pd.read_csv(
    PATH_DATA + "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    PATH_DATA + "olist_order_reviews_dataset.csv"
)

sellers = pd.read_csv(
    PATH_DATA + "olist_sellers_dataset.csv"
)

geolocation = pd.read_csv(
    PATH_DATA + "olist_geolocation_dataset.csv"
)

## 1. Perguntas iniciais de entendimento dos dados

### 1.1 Estrutura das tabelas

- Quantas linhas existem em cada tabela?
- Quais colunas existem em cada tabela?
- Qual é a granularidade de cada tabela?
  - `orders`: uma linha representa o quê?
  - `order_items`: uma linha representa o quê?
  - `customers`: uma linha representa o quê?
  - `reviews`: uma linha representa o quê?
  - `payments`: uma linha representa o quê?

### 1.3 Pedidos e sellers

- Um mesmo pedido pode conter produtos de mais de um seller?
- Quantos pedidos possuem mais de um seller?
- Qual percentual dos pedidos possui mais de um seller?

### 1.4 Clientes e sellers

- Um mesmo cliente pode comprar de mais de um seller ao longo do tempo?
- Quantos clientes compraram de mais de um seller?
- Qual percentual dos clientes comprou de mais de um seller?

### 1.5 Avaliações

- Todo pedido possui avaliação?
- Existe pedido com mais de uma avaliação?
- A tabela `reviews` pode ser analisada no nível de pedido?

### 1.6 Pagamentos

- Todo pedido possui pagamento?
- Existe pedido com mais de uma forma ou parcela de pagamento?
- A tabela `payments` está no nível de pedido ou no nível de transação de pagamento?

### 1.7 Unidade de análise

Com base nas respostas anteriores, precisamos definir:

- Qual será a unidade de análise para satisfação do cliente?
- Qual será a unidade de análise para análise de sellers?
- Qual será a unidade de análise para análise de produtos e categorias?


In [ ]:
print("sellers: ", sellers.shape)

print("customers: ", customers.shape)

print("orders: ", orders.shape)

print("order_items: ", order_items.shape)

print("products: ", products.shape)

print("reviews: ", reviews.shape)

print("geolocation: ", geolocation.shape)

print("payments: ", payments.shape)

Visão inicial do DataFrame

In [ ]:
sellers.head()

In [ ]:
customers.head()

In [ ]:
orders.head()

In [ ]:
order_items.head()

In [ ]:
products.head()

In [ ]:
reviews.head()

In [ ]:
geolocation.head()

In [ ]:
payments.head()

Visão info dos DataFrames

In [ ]:
sellers.info()

In [ ]:
customers.info()

In [ ]:
orders.info()

In [ ]:
order_items.info()

In [ ]:
products.info()

In [ ]:
reviews.info()

In [ ]:
geolocation.info()

In [ ]:
payments.info()

### 1.2 Clientes

- Qual é a diferença entre `customer_id` e `customer_unique_id`?
- Quantos `customer_id` existem?
- Quantos `customer_unique_id` existem?
- Existem clientes com mais de um pedido?

In [44]:
print("customer_id: ",customers["customer_id"].nunique())
print("customer_unique_id: ",customers["customer_unique_id"].nunique())

customer_id:  99441
customer_unique_id:  96096


In [ ]:
order_customers = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

order_customers.head()

In [ ]:
orders_per_customer = (
    order_customers
    .groupby("customer_unique_id")
    .agg(qtd_pedidos = ("order_id", "nunique"))
    .reset_index()
)

orders_per_customer.head()

In [39]:
print("Quantidade de clientes com mais de um pedido: ",(orders_per_customer["qtd_pedidos"] > 1).sum())

Quantidade de clientes com mais de um pedido:  2997


In [ ]:
distribuicao_pedidos = (
    orders_per_customer["qtd_pedidos"]
    .value_counts()
    .sort_index()
    .reset_index()
)

distribuicao_pedidos.columns = ["qtd_pedidos", "qtd_clientes"]

distribuicao_pedidos

In [41]:
((99441-96096)/99441)*100

3.3638036624732255

Ao fazermos essa analise inicial de distribuição de pedidos por clientes, observamos que apenas 3%, aproximadamente, da base realizou mais de uma compra.

### 1.3 Pedidos e sellers

- Um mesmo pedido pode conter produtos de mais de um seller?
- Quantos pedidos possuem mais de um seller?
- Qual percentual dos pedidos possui mais de um seller?

In [42]:
orders_sellers = (
    order_items.groupby("order_id")
    .agg(
        qtd_itens = ("order_item_id", "count"),
        qtd_produtos = ("product_id", "nunique"),
        qtd_sellers = ("seller_id", "nunique")
    )
    .reset_index()
)

orders_sellers.sort_values("qtd_sellers", ascending=False).head(10)

,order_id,qtd_itens,qtd_produtos,qtd_sellers
10831,1c11d0f4353b31ac3417fbfa5f0f2a8a,7,6,5
79967,cf5c8d9f52807cb2d2f0a0ff54c478da,6,6,5
55847,91be51c856a90d7efe86cf9d082d6ae3,4,4,4
11231,1d23106803c48c391366ff224513fb7f,4,4,4
53796,8c2b13adf3f377c8f2b06b04321b0925,4,4,4
45702,76c4c846aae2dae9e87dfa492c3f5259,3,3,3
46192,780dfcd5aaf2662f3f6ce2201164394b,4,4,3
8060,14f0429b74bbb862693e5a88af6f6f6a,3,3,3
19793,338ffe54b65a3b8124c81ebe6d1cc4b0,3,3,3
20349,34fdc362961364d3ff08986ccff2212d,3,3,3


In [43]:
customers_sellers = (
    order_items
    .merge(orders[["order_id", "customer_id"]], on="order_id", how="left")
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
    .groupby("customer_unique_id")
    .agg(
        qtd_pedidos=("order_id", "nunique"),
        qtd_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

customers_sellers.sort_values("qtd_pedidos", ascending=False).head(10)

,customer_unique_id,qtd_pedidos,qtd_sellers
52597,8d50f5eadf50201ccdcedfb9e2ac8455,16,8
23302,3e43e6105506432c953e165fb2acf44c,9,10
10281,1b6c7548a2a1f9037c1fd3ddfed95f33,7,6
37532,6469f99c1f9dfae7733b25662e7f1782,7,4
75560,ca77025e7201e3b30c44b472ff346268,7,6
26849,47c1a3033b8b77b3ab6e109eb4d5fdf3,6,6
7124,12f5d6e1cbf93dafd9dcc19095df0b3d,6,1
37320,63cfc61cee11cbe306bff5857d00bfe4,6,5
82305,dc813062e0fc23409cd255f7f53c7074,6,6
89814,f0e310a6839dce9de1638e0fe5ab282a,6,6
